# GFM Book Text-to-Speech Pipeline

Converts Quarto book chapters to audio using Google Cloud TTS.

## Prerequisites

**Google Cloud:**
- Google Cloud account with Text-to-Speech API enabled
- `gcloud` CLI installed and authenticated

**System dependencies:**
- `pandoc` - for converting Quarto to plain text
- `ffmpeg` - for audio concatenation (avoids chunk boundary artifacts)

**Install ffmpeg:**
```bash
# macOS
brew install ffmpeg

# Ubuntu/Debian
sudo apt-get install ffmpeg

# Windows (with chocolatey)
choco install ffmpeg

# Conda
conda install -c conda-forge ffmpeg
```

**Cost:** ~$30/1M chars for Chirp3-HD voices. A typical chapter (~60K chars) costs ~$1.80.

## 1. Setup

In [24]:
# Install Python dependencies
# !pip install -q google-cloud-texttospeech

# Check for system dependencies
import shutil

deps_ok = True
for cmd in ["pandoc", "ffmpeg"]:
    if shutil.which(cmd):
        print(f"✓ {cmd} found")
    else:
        print(f"✗ {cmd} NOT FOUND - please install (see instructions above)")
        deps_ok = False

if not deps_ok:
    print(
        "\n⚠️  Missing dependencies will cause errors. Install them before proceeding."
    )

✓ pandoc found
✓ ffmpeg found


In [9]:
# Authenticate with GCP (run once, follow the browser prompt)
# !gcloud auth application-default login

In [25]:
import re
import subprocess
from pathlib import Path
from google.cloud import texttospeech

ImportError: cannot import name 'texttospeech' from 'google.cloud' (unknown location)

In [47]:
# Import pronunciation guide (same directory as notebook)
try:
    from pronunciation_guide import apply_pronunciations

    PRONUNCIATIONS_AVAILABLE = True
    print("Pronunciation guide loaded.")
except ImportError:
    PRONUNCIATIONS_AVAILABLE = False

    def apply_pronunciations(text, verbose=False):
        return text

    print("Warning: pronunciation_guide.py not found, skipping term corrections.")

print("Setup complete!")

Pronunciation guide loaded.
Setup complete!


# Path to gfm-book repository
BOOK_ROOT = Path("..")  # Update this path as needed

# Chapter to convert (update for different chapters)
CHAPTER_FILE = BOOK_ROOT / "part_2" / "p2-ch05-representations.qmd"
# CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch28-clinical-risk.qmd"
# CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch29-rare-disease.qmd"

# Output directory
OUTPUT_DIR = Path(".")  # Current directory, or set to BOOK_ROOT / "audio"
OUTPUT_DIR.mkdir(exist_ok=True)

# === VOICE CONFIGURATION ===
# Three voices for different content types:
# - VOICE_BODY: Main content (most of the text)
# - VOICE_HEADING: Section headings (signals transitions)
# - VOICE_ASIDE: Callouts, notes, block quotes (supplementary content)

# Chirp3-HD voices ($30/1M chars) - most natural
# Male options: Charon, Fenrir, Kore, Orus, Puck
# Female options: Achernar, Aoede, Leda, Schedar, Sulafat, Zephyr
VOICE_BODY = "en-US-Chirp3-HD-Charon"      # Male - main narrator
VOICE_HEADING = "en-US-Chirp3-HD-Aoede"    # Female - section transitions  
VOICE_ASIDE = "en-US-Chirp3-HD-Kore"       # Different male - asides/notes

# Alternative: all same gender
# VOICE_BODY = "en-US-Chirp3-HD-Charon"
# VOICE_HEADING = "en-US-Chirp3-HD-Fenrir"
# VOICE_ASIDE = "en-US-Chirp3-HD-Puck"

# Speaking rate: 0.25 to 2.0 (1.0 = normal)
SPEAKING_RATE = 1.25  # Slightly faster, good for technical content

# Audio profile optimized for listening device
AUDIO_PROFILE = "headphone-class-device"

# Use SSML for voice switching and pauses
USE_SSML = True

# Apply pronunciation corrections for genomics/ML terms
APPLY_PRONUNCIATIONS = True

print(f"Chapter: {CHAPTER_FILE.name}")
print(f"Voices:")
print(f"  Body: {VOICE_BODY}")
print(f"  Headings: {VOICE_HEADING}")
print(f"  Asides: {VOICE_ASIDE}")
print(f"Rate: {SPEAKING_RATE}x")
print(f"Audio profile: {AUDIO_PROFILE}")

In [48]:
# Path to gfm-book repository
BOOK_ROOT = Path("..")  # Update this path as needed

# Chapter to convert (update for different chapters)
# CHAPTER_FILE = BOOK_ROOT / "part_2" / "p2-ch05-representations.qmd"
# CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch28-clinical-risk.qmd"
CHAPTER_FILE = BOOK_ROOT / "part_7" / "p7-ch29-rare-disease.qmd"

# Output directory
OUTPUT_DIR = Path(".")  # Current directory, or set to BOOK_ROOT / "audio"
OUTPUT_DIR.mkdir(exist_ok=True)

# Voice options
# Neural2/WaveNet voices ($16/1M chars)
# VOICE_NAME = "en-US-Neural2-D"  # Male, natural
# VOICE_NAME = "en-US-Neural2-F"  # Female, natural
# VOICE_NAME = "en-US-Wavenet-D"  # Male, good balance

# Chirp3-HD voices ($30/1M chars) - most natural
VOICE_NAME = "en-US-Chirp3-HD-Charon"
# Other Chirp3-HD: Achernar, Aoede, Kore, Leda, Puck, Schedar, Zephyr

# Studio voices ($160/1M chars) - highest quality
# VOICE_NAME = "en-US-Studio-O"

# Speaking rate: 0.25 to 2.0 (1.0 = normal)
SPEAKING_RATE = 1.35  # Slightly faster, good for technical content

# Audio profile optimized for listening device
# Options: headphone-class-device, handset-class-device, small-bluetooth-speaker-class-device
AUDIO_PROFILE = "headphone-class-device"

# Use SSML for natural pauses at section headings and paragraphs
USE_SSML = True

# Apply pronunciation corrections for genomics/ML terms
APPLY_PRONUNCIATIONS = True

print(f"Chapter: {CHAPTER_FILE.name}")
print(f"Voice: {VOICE_NAME}")
print(f"Rate: {SPEAKING_RATE}x")
print(f"Audio profile: {AUDIO_PROFILE}")
print(f"SSML breaks: {'enabled' if USE_SSML else 'disabled'}")
print(f"Pronunciations: {'enabled' if APPLY_PRONUNCIATIONS else 'disabled'}")

Chapter: p7-ch29-rare-disease.qmd
Voice: en-US-Chirp3-HD-Charon
Rate: 1.35x
Audio profile: headphone-class-device
SSML breaks: enabled
Pronunciations: enabled


In [ ]:
def convert_qmd_to_text(qmd_path: Path) -> str:
    """Convert Quarto file to plain text using pandoc."""
    result = subprocess.run(
        ["pandoc", str(qmd_path), "-t", "plain", "--wrap=none"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Pandoc failed: {result.stderr}")
    return result.stdout


def extract_headings_from_qmd(qmd_path: Path) -> set:
    """Extract heading text from .qmd file to identify them later."""
    headings = set()
    with open(qmd_path, 'r', encoding='utf-8') as f:
        for line in f:
            match = re.match(r'^#{1,6}\s+(.+?)(?:\s*\{[^}]*\})?\s*$', line)
            if match:
                heading_text = match.group(1).strip()
                heading_text = re.sub(r'\*\*([^*]+)\*\*', r'\1', heading_text)
                heading_text = re.sub(r'\*([^*]+)\*', r'\1', heading_text)
                heading_text = re.sub(r'`([^`]+)`', r'\1', heading_text)
                headings.add(heading_text)
    return headings


def extract_callout_content(qmd_path: Path) -> set:
    """Extract text that appears inside callout blocks."""
    callout_lines = set()
    in_callout = False
    
    with open(qmd_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Start of callout block
            if re.match(r'^:::\s*\{\.callout-', line):
                in_callout = True
                continue
            # End of callout block
            if in_callout and line.strip() == ':::':
                in_callout = False
                continue
            # Capture content inside callout (first 50 chars for matching)
            if in_callout and line.strip():
                # Clean and store for matching
                clean = line.strip()
                clean = re.sub(r'\*\*([^*]+)\*\*', r'\1', clean)
                clean = re.sub(r'\*([^*]+)\*', r'\1', clean)
                clean = re.sub(r'`([^`]+)`', r'\1', clean)
                if len(clean) > 10:  # Only meaningful content
                    callout_lines.add(clean[:80])  # First 80 chars for matching
    
    return callout_lines


def convert_table_to_prose(table_text: str) -> str:
    """Convert a markdown/plain table to speakable prose."""
    lines = table_text.strip().split('\n')
    content_lines = [l for l in lines if not re.match(r'^[\s\-\|:]+$', l)]
    
    if len(content_lines) < 2:
        return table_text
    
    header_line = content_lines[0]
    headers = [h.strip() for h in header_line.split('|') if h.strip()]
    
    if len(headers) < 2:
        return table_text
    
    prose_parts = []
    for row_line in content_lines[1:]:
        cells = [c.strip() for c in row_line.split('|') if c.strip()]
        
        if len(cells) != len(headers):
            continue
        
        row_prose = f"For {cells[0]}: "
        details = []
        for i, (header, value) in enumerate(zip(headers[1:], cells[1:])):
            if value and value != '-':
                header_clean = header.lower().replace('_', ' ')
                details.append(f"{header_clean} is {value}")
        
        if details:
            row_prose += ", ".join(details) + "."
            prose_parts.append(row_prose)
    
    if prose_parts:
        return "\n".join(prose_parts)
    return table_text


# Content type markers (replaced with voice tags later)
MARKER_HEADING = "___HEADING___"
MARKER_ASIDE = "___ASIDE___"
MARKER_BODY = "___BODY___"
MARKER_END = "___END___"


def preprocess_for_tts(text: str, headings: set = None, callout_content: set = None) -> str:
    """Clean text for TTS consumption. Returns plain text with content markers.
    
    Markers are added for different content types:
    - ___HEADING___...___END___ for section headings
    - ___ASIDE___...___END___ for callouts/notes
    - ___BODY___...___END___ for regular content
    """
    if headings is None:
        headings = set()
    if callout_content is None:
        callout_content = set()

    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F\u00C0-\u00FF\u2018\u2019\u201C\u201D\u2013\u2014]+', ' ', text)

    # Convert tables to prose
    table_pattern = r'((?:^[|\s].*\n)+)'
    def table_replacer(match):
        table_text = match.group(1)
        if '|' in table_text and re.search(r'^[\s\-\|:]+$', table_text, re.MULTILINE):
            return convert_table_to_prose(table_text) + "\n"
        return table_text
    text = re.sub(table_pattern, table_replacer, text, flags=re.MULTILINE)
    
    # Handle pandoc plain-text tables
    dash_table_pattern = r'(^.+\n)(^[\-\s]+$\n)((?:^.+\n?)+)'
    def dash_table_replacer(match):
        header_line = match.group(1).strip()
        data_lines = match.group(3).strip().split('\n')
        headers = re.split(r'\s{2,}', header_line)
        headers = [h.strip() for h in headers if h.strip()]
        if len(headers) < 2:
            return match.group(0)
        prose_parts = []
        for row in data_lines:
            cells = re.split(r'\s{2,}', row)
            cells = [c.strip() for c in cells if c.strip()]
            if len(cells) >= len(headers):
                row_prose = f"For {cells[0]}: "
                details = []
                for header, value in zip(headers[1:], cells[1:]):
                    if value and value != '-':
                        details.append(f"{header.lower()} is {value}")
                if details:
                    row_prose += ", ".join(details) + "."
                    prose_parts.append(row_prose)
        if prose_parts:
            return "\n".join(prose_parts) + "\n"
        return match.group(0)
    text = re.sub(dash_table_pattern, dash_table_replacer, text, flags=re.MULTILINE)

    # Remove math
    text = re.sub(r"\$\$.*?\$\$", " [Equation omitted] ", text, flags=re.DOTALL)
    text = re.sub(r"\$[^$]+\$", "", text)

    # DROP cross-references and citations
    text = re.sub(r"@sec-ch\d+-[\w-]+", "", text)
    text = re.sub(r"@fig-[\w-]+", "", text)
    text = re.sub(r"@tbl-[\w-]+", "", text)
    text = re.sub(r"@eq-[\w-]+", "", text)
    text = re.sub(r"\[@[\w_-]+(?:;\s*@[\w_-]+)*\]", "", text)
    text = re.sub(r"@[\w_-]+", "", text)

    # DROP figures
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", "", text)
    text = re.sub(r"^\s*\[Figure[^\]]*\].*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^Figure \d+[.:][^\n]*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*Figure\s+\d+\s*$", "", text, flags=re.MULTILINE)

    # Mark callout labels but keep content (will be tagged as ASIDE)
    text = re.sub(r":::\s*\{\.callout-(\w+)[^}]*\}", r"[\1]:", text)
    text = re.sub(r":::", "", text)

    # Remove layout directives
    text = re.sub(r"\{#[\w-]+[^}]*\}", "", text)
    text = re.sub(r"\{layout[^}]*\}", "", text)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    text = re.sub(r"```[\s\S]*?```", " [Code block omitted] ", text)

    # Clean markdown
    text = re.sub(r"^#+\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"\*\*([^*]+)\*\*", r"\1", text)
    text = re.sub(r"\*([^*]+)\*", r"\1", text)
    text = re.sub(r"__([^_]+)__", r"\1", text)
    text = re.sub(r"_([^_]+)_", r"\1", text)
    text = re.sub(r"`([^`]+)`", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"^\s*[-*+]\s+", "  ", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\.\s+", "  ", text, flags=re.MULTILINE)
    text = re.sub(r"^.*Estimated reading time.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^[\-_\s]{3,}$", "", text, flags=re.MULTILINE)

    # Normalize whitespace
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' \n', '\n', text)
    text = re.sub(r'\n ', '\n', text)

    # Process lines and add content type markers
    lines = text.strip().split('\n')
    processed_lines = []
    in_aside = False
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        
        # Check for callout/note markers
        if re.match(r'^\[(note|tip|warning|important|caution)\]:', stripped, re.IGNORECASE):
            in_aside = True
            # Keep the label but mark as aside
            stripped = re.sub(r'^\[(\w+)\]:', r'\1:', stripped)
        
        # Check if line matches callout content from source
        is_callout_content = any(stripped[:80].startswith(c[:40]) for c in callout_content if len(c) > 10)
        
        # Check if heading
        is_heading = stripped in headings or stripped.rstrip('.') in headings
        
        # Add period if needed
        if stripped and not stripped.endswith(('.', '?', '!', ':', ';', ',')):
            stripped += '.'
        
        # Apply markers
        if is_heading:
            processed_lines.append(f'{MARKER_HEADING}{stripped}{MARKER_END}')
            in_aside = False  # Reset after heading
        elif in_aside or is_callout_content:
            processed_lines.append(f'{MARKER_ASIDE}{stripped}{MARKER_END}')
        else:
            processed_lines.append(f'{MARKER_BODY}{stripped}{MARKER_END}')
    
    return '\n'.join(processed_lines)


def build_ssml(text: str, voice_body: str, voice_heading: str, voice_aside: str) -> str:
    """Convert marked-up text to SSML with voice tags."""
    
    # Escape XML special chars first (before adding SSML tags)
    text = text.replace('&', '&amp;')
    text = text.replace('<', '&lt;')
    text = text.replace('>', '&gt;')
    
    # Replace markers with SSML voice tags
    # Heading: break + voice + break
    text = text.replace(
        MARKER_HEADING, 
        f'<break time="1s"/><voice name="{voice_heading}">'
    )
    text = text.replace(
        MARKER_ASIDE,
        f'<voice name="{voice_aside}">'
    )
    text = text.replace(
        MARKER_BODY,
        f'<voice name="{voice_body}">'
    )
    
    # End markers become voice close + break for headings
    # We need to track which type we're closing - simpler to just close voice
    text = text.replace(MARKER_END, '</voice>')
    
    # Add break after headings (find heading close and add break)
    text = re.sub(
        rf'(<voice name="{re.escape(voice_heading)}">[^<]+</voice>)',
        r'\1<break time="500ms"/>',
        text
    )
    
    return f'<speak>\n{text}\n</speak>'


print("Preprocessing functions defined.")

## 4. TTS Generation Functions

In [ ]:
import tempfile
import os

def split_marked_text_into_chunks(text: str, max_bytes: int = 4000) -> list:
    """Split marked text into chunks that fit within API byte limits.
    
    Splits on line boundaries to keep content markers intact.
    """
    chunks = []
    current_chunk = []
    current_size = 0
    
    lines = text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        line_size = len(line.encode('utf-8'))
        
        # Estimate SSML overhead per line (~100 bytes for voice tags)
        estimated_size = line_size + 100
        
        if current_size + estimated_size > max_bytes and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = []
            current_size = 0
        
        current_chunk.append(line)
        current_size += estimated_size
    
    if current_chunk:
        chunks.append('\n'.join(current_chunk))
    
    return chunks


def generate_audio(
    marked_text: str,
    output_path: Path,
    voice_body: str,
    voice_heading: str,
    voice_aside: str,
    speaking_rate: float,
    audio_profile: str = None,
) -> None:
    """Generate audio from marked text using Google Cloud TTS with multiple voices.
    
    Builds SSML per-chunk to ensure valid voice tag structure.
    """
    client = texttospeech.TextToSpeechClient()

    # Split marked text (before SSML conversion)
    chunks = split_marked_text_into_chunks(marked_text)
    total_chars = sum(len(c) for c in chunks)
    print(f"Processing {len(chunks)} chunks ({total_chars:,} characters)...")

    # Default voice (SSML voice tags override as needed)
    voice = texttospeech.VoiceSelectionParams(language_code="en-US", name=voice_body)
    
    audio_config_params = {
        "audio_encoding": texttospeech.AudioEncoding.LINEAR16,
        "sample_rate_hertz": 24000,
        "speaking_rate": speaking_rate,
    }
    
    if audio_profile:
        audio_config_params["effects_profile_id"] = [audio_profile]
    
    if "Chirp" not in voice_body:
        audio_config_params["pitch"] = 0.0
    
    audio_config = texttospeech.AudioConfig(**audio_config_params)

    with tempfile.TemporaryDirectory() as tmpdir:
        chunk_files = []
        
        for i, chunk in enumerate(chunks):
            print(f"  Chunk {i+1}/{len(chunks)}...", end=" ", flush=True)
            
            # Build SSML for this chunk
            ssml = build_ssml(chunk, voice_body, voice_heading, voice_aside)
            
            synthesis_input = texttospeech.SynthesisInput(ssml=ssml)
            
            response = client.synthesize_speech(
                input=synthesis_input, voice=voice, audio_config=audio_config
            )
            
            chunk_path = os.path.join(tmpdir, f"chunk_{i:04d}.wav")
            with open(chunk_path, "wb") as f:
                f.write(response.audio_content)
            chunk_files.append(chunk_path)
            print("done")

        print("\nConcatenating and converting to MP3...")
        
        list_path = os.path.join(tmpdir, "files.txt")
        with open(list_path, "w") as f:
            for chunk_path in chunk_files:
                f.write(f"file '{chunk_path}'\n")
        
        output_str = str(output_path)
        result = subprocess.run([
            "ffmpeg", "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", list_path,
            "-codec:a", "libmp3lame",
            "-qscale:a", "2",
            output_str
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"ffmpeg error: {result.stderr}")
            raise RuntimeError("ffmpeg concatenation failed")

    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"Saved: {output_path} ({size_mb:.1f} MB)")


print("TTS functions defined.")

## 5. Run the Pipeline

In [ ]:
# Step 1: Convert Quarto to plain text and extract content markers
print(f"Converting {CHAPTER_FILE.name}...")
raw_text = convert_qmd_to_text(CHAPTER_FILE)
headings = extract_headings_from_qmd(CHAPTER_FILE)
callout_content = extract_callout_content(CHAPTER_FILE)

print(f"  Raw text: {len(raw_text):,} characters")
print(f"  Found {len(headings)} headings")
print(f"  Found {len(callout_content)} callout lines")

In [ ]:
# Step 2: Preprocess for TTS
print("Preprocessing...")

# Clean text and add content type markers
marked_text = preprocess_for_tts(raw_text, headings=headings, callout_content=callout_content)

# Apply pronunciation guide (on marked text, before SSML conversion)
if APPLY_PRONUNCIATIONS:
    print("Applying pronunciation guide...")
    marked_text = apply_pronunciations(marked_text, verbose=True)

# Count content types
heading_count = marked_text.count(MARKER_HEADING)
aside_count = marked_text.count(MARKER_ASIDE)
body_count = marked_text.count(MARKER_BODY)

print(f"\n  Marked text: {len(marked_text):,} characters")
print(f"  Content breakdown:")
print(f"    Headings: {heading_count} lines ({VOICE_HEADING})")
print(f"    Asides: {aside_count} lines ({VOICE_ASIDE})")
print(f"    Body: {body_count} lines ({VOICE_BODY})")

In [ ]:
# Preview the marked text and SSML
print("=" * 60)
print("MARKED TEXT PREVIEW (first 1500 chars):")
print("=" * 60)
print(marked_text[:1500])

print("\n" + "=" * 60)
print("SSML PREVIEW (first chunk):")
print("=" * 60)
sample_ssml = build_ssml(marked_text[:1500], VOICE_BODY, VOICE_HEADING, VOICE_ASIDE)
print(sample_ssml[:2000])

In [54]:
def estimate_tts_cost(char_count: int, voice_name: str) -> float:
    """Estimate cost in USD for Google Cloud TTS."""
    # Pricing per 1M characters (as of 2025)
    # https://cloud.google.com/text-to-speech/pricing
    if "Studio" in voice_name:
        rate = 160.00  # Studio voices
    elif "Chirp" in voice_name:
        rate = 30.00  # Chirp3-HD voices
    elif "Neural2" in voice_name or "Wavenet" in voice_name:
        rate = 16.00  # Neural2/WaveNet
    else:
        rate = 4.00  # Standard voices

    return (char_count / 1_000_000) * rate


total_chars = len(clean_text)
cost = estimate_tts_cost(total_chars, VOICE_NAME)
print(f"Processing ({total_chars:,} characters, ~${cost:.2f})...")

Processing (57,531 characters, ~$1.73)...


In [ ]:
# Step 3: Generate audio
chapter_name = CHAPTER_FILE.stem.replace(".", "-")
output_file = OUTPUT_DIR / f"{chapter_name}.mp3"

print(f"Generating audio...")
print(f"  Voices: {VOICE_BODY} (body), {VOICE_HEADING} (headings), {VOICE_ASIDE} (asides)")
print(f"  Speaking rate: {SPEAKING_RATE}x")
print(f"  Audio profile: {AUDIO_PROFILE}")

# Pass marked text (not SSML) - SSML is built per-chunk in generate_audio
generate_audio(
    marked_text, 
    output_file, 
    VOICE_BODY, 
    VOICE_HEADING, 
    VOICE_ASIDE,
    SPEAKING_RATE, 
    AUDIO_PROFILE
)

In [55]:
# Step 3: Generate audio
chapter_name = CHAPTER_FILE.stem.replace(".", "-")
output_file = OUTPUT_DIR / f"{chapter_name}.mp3"

print(f"Generating audio with voice: {VOICE_NAME}")
print(f"Speaking rate: {SPEAKING_RATE}x, Audio profile: {AUDIO_PROFILE}")
generate_audio(trimmed_text, output_file, VOICE_NAME, SPEAKING_RATE, AUDIO_PROFILE)

Generating audio with voice: en-US-Chirp3-HD-Charon
Speaking rate: 1.35x, Audio profile: headphone-class-device


NameError: name 'texttospeech' is not defined

## 6. Batch Processing (Optional)

Generate audio for multiple chapters at once.

In [ ]:
# List available chapters
chapters = sorted(BOOK_ROOT.glob("part_*/p*-ch*.qmd"))
print(f"Found {len(chapters)} chapters:")
for i, ch in enumerate(chapters[:10]):
    print(f"  {i+1}. {ch.relative_to(BOOK_ROOT)}")
if len(chapters) > 10:
    print(f"  ... and {len(chapters) - 10} more")

In [ ]:
# Batch convert selected chapters (uncomment and modify as needed)
# WARNING: This will use API quota for each chapter

# chapters_to_convert = [
#     BOOK_ROOT / "part_2" / "p2-ch05-representations.qmd",
#     BOOK_ROOT / "part_2" / "p2-ch06-cnns.qmd",
#     BOOK_ROOT / "part_2" / "p2-ch07-attention.qmd",
# ]

# for chapter in chapters_to_convert:
#     print(f"\n{'='*60}")
#     print(f"Processing: {chapter.name}")
#     print(f"{'='*60}")
#
#     raw = convert_qmd_to_text(chapter)
#     clean = preprocess_for_tts(raw)
#     output = OUTPUT_DIR / f"{chapter.stem}.mp3"
#     generate_audio(clean, output, VOICE_NAME, SPEAKING_RATE)

## 7. Voice Comparison (Optional)

Generate samples with different voices to compare quality.

In [ ]:
# Sample text for voice comparison
sample_text = clean_text[:3000]  # First 3000 chars

voices_to_test = [
    "en-US-Neural2-D",  # Male, natural
    "en-US-Neural2-F",  # Female, natural
    # "en-US-Studio-O",   # Male, studio (highest quality)
]

# Uncomment to generate samples
# for voice in voices_to_test:
#     output = OUTPUT_DIR / f"sample_{voice}.mp3"
#     print(f"\nGenerating sample with {voice}...")
#     generate_audio(sample_text, output, voice, SPEAKING_RATE)

---

## Reference

### Voices

| Voice Type | Example | Cost/1M chars |
|------------|---------|---------------|
| Standard | en-US-Standard-D | $4 |
| Neural2/WaveNet | en-US-Neural2-D | $16 |
| **Chirp3-HD** | en-US-Chirp3-HD-Charon | **$30** |
| Studio | en-US-Studio-O | $160 |

**Chirp3-HD voices:** Achernar, Achird, Algenib, Aoede, Charon, Kore, Leda, Puck, Schedar, Zephyr, and more.

### Audio Profiles

| Profile | Optimized For |
|---------|---------------|
| `headphone-class-device` | Earbuds, headphones (recommended) |
| `handset-class-device` | Smartphones |
| `small-bluetooth-speaker-class-device` | Portable speakers |
| `large-automotive-class-device` | Car speakers |

### AudioConfig Parameters

| Parameter | Range | Notes |
|-----------|-------|-------|
| `speaking_rate` | 0.25 - 2.0 | 1.0 = normal, 1.25-1.5 recommended |
| `pitch` | -20.0 - 20.0 | Semitones (not supported for Chirp) |
| `volume_gain_db` | -96.0 - 16.0 | Max +10 dB recommended |

### Links

- [Chirp3-HD Voices](https://cloud.google.com/text-to-speech/docs/chirp3-hd)
- [Audio Profiles](https://cloud.google.com/text-to-speech/docs/audio-profiles)
- [Pricing](https://cloud.google.com/text-to-speech/pricing)
- [Python API Reference](https://cloud.google.com/python/docs/reference/texttospeech/latest)